In [0]:
create catalog if not exists deltalake
comment "This is catalog for all the tables in the lakehouse"

In [0]:
create  schema if not exists deltalake.streammetadata
comment "This is schema holds  stream metadata  ";

create  schema if not exists deltalake.bronze
comment "This is schema holds  stream bronzedata  ";

create  schema if not exists deltalake.silver
comment "This is schema holds  stream silver data  ";


create  schema if not exists deltalake.gold
comment "This is schema holds  stream gold data  ";



In [0]:
create external volume if not exists deltalake.streammetadata.autoloader
location 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/almetadata/'
comment 'Metedata for autoloader ';

In [0]:


create external volume if not exists deltalake.streammetadata.chckpoint
location 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/checkpoint/'
comment 'checkpointlocation ';

In [0]:
create external volume if not exists deltalake.bronze.rawinbound
location 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/data/'
comment 'inbound location ';

In [0]:
%python
df_raw = (
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/deltalake/streammetadata/autoloader")
    .load("/Volumes/filestreaming/fakerstream/inbound")
)



In [0]:
%python

query = (
  df_raw.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/deltalake/streammetadata/chckpoint")
    .outputMode("append")
    .start("/Volumes/deltalake/bronze/rawinbound")
)

In [0]:
%python
query.lastProgress